Analisis Eksploratif dan Evaluasi Model Machine Learning untuk Prediksi Kualitas dan Klasifikasi Jenis Wine Berdasarkan Karakteristik Fisikokimia
# 01 — Data Preparation: Wine Quality Dataset

## 1. Deskripsi

Notebook ini digunakan untuk melakukan tahap *data preparation* pada dataset Wine Quality yang terdiri dari data red wine dan white wine. Tahapan ini bertujuan untuk memahami struktur data, memeriksa kualitas data, menggabungkan kedua dataset, serta melakukan pembersihan data sebelum digunakan dalam analisis dan pemodelan Machine Learning.

## 2. Tujuan

Tujuan dari tahap data preparation ini adalah:

1. Memuat dan memahami struktur dataset red wine dan white wine.
2. Memeriksa tipe data, missing values, duplikat, serta potensi nilai yang tidak valid.
3. Menggabungkan kedua dataset dan menambahkan variabel `wine_type` sebagai label jenis wine.
4. Menganalisis distribusi awal variabel `quality` dan `wine_type`.
5. Mengevaluasi dampak penghapusan duplikat terhadap jumlah dan distribusi data.
6. Melakukan validasi serta menyimpan dataset hasil cleaning.

## 3. Dataset

Dataset yang digunakan terdiri dari dua sumber:

- `winequality-red.csv`: data karakteristik kimia red wine.
- `winequality-white.csv`: data karakteristik kimia white wine.

Kedua dataset memiliki variabel karakteristik kimia yang sama dan variabel target `quality`. Setelah digabungkan, ditambahkan variabel `wine_type` dengan ketentuan:

| Nilai | Jenis wine |
|---:|---|
| 0 | Red wine |
| 1 | White wine |

Dataset gabungan memiliki 13 kolom, yang terdiri dari 11 variabel karakteristik kimia, 1 variabel `quality`, dan 1 variabel `wine_type`.

## 4. Tahapan Pengerjaan

Tahapan yang dilakukan dalam notebook ini meliputi:

1. **Data Loading** — memuat dataset dari file CSV.
2. **Initial Data Inspection** — memeriksa struktur, tipe data, dan kualitas awal dataset.
3. **Data Integration** — menggabungkan dataset red wine dan white wine.
4. **Data Quality Audit** — memeriksa missing values, duplikat, nilai tak hingga, dan distribusi variabel.
5. **Data Cleaning** — mengevaluasi dan menangani duplikat berdasarkan hasil audit.
6. **Data Validation** — memeriksa kembali dataset setelah cleaning.
7. **Data Export** — menyimpan dataset hasil cleaning untuk digunakan pada tahap berikutnya.

## 5. Output

Output dari notebook ini adalah dataset hasil cleaning yang disimpan dalam format CSV pada lokasi:

`data/processed/winequality_cleaned.csv`

Dataset tersebut akan digunakan sebagai masukan untuk tahap *Exploratory Data Analysis* (EDA) dan eksperimen Machine Learning berikutnya.

## 6. Catatan

Penghapusan duplikat dilakukan setelah mengevaluasi kondisi dan dampaknya terhadap distribusi data. Dataset asli tetap dipertahankan sebagai sumber agar proses preprocessing dapat ditinjau dan direproduksi kembali.

In [ ]:
# Import library untuk audit dataset

from pathlib import Path

import pandas as pd
import numpy as np

# Pengaturan tampilan pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.precision", 4)

print("Library berhasil diimpor.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Library berhasil diimpor.
Pandas version: 3.0.5
NumPy version: 2.5.2


In [ ]:
# Menentukan direktori proyek secara otomatis

PROJECT_ROOT = Path.cwd()

# Jika notebook dijalankan dari folder notebooks,
# direktori proyek berada satu tingkat di atasnya.
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

RED_PATH = DATA_DIR / "winequality-red.csv"
WHITE_PATH = DATA_DIR / "winequality-white.csv"

print("Direktori proyek :", PROJECT_ROOT)
print("Direktori data   :", DATA_DIR)
print("File red wine   :", RED_PATH.exists())
print("File white wine :", WHITE_PATH.exists())

In [ ]:
# Membaca dataset sumber

df_red_raw = pd.read_csv(RED_PATH, sep=";")
df_white_raw = pd.read_csv(WHITE_PATH, sep=";")

print("Dataset berhasil dibaca.")

In [ ]:
# Melihat lima baris pertama dari masing-masing dataset

print("RED WINE")
display(df_red_raw.head())

print("WHITE WINE")
display(df_white_raw.head())

In [ ]:
# Audit ukuran dan struktur dataset

print("RED WINE")
print("Shape:", df_red_raw.shape)
print("Columns:", df_red_raw.columns.tolist())

print("\nWHITE WINE")
print("Shape:", df_white_raw.shape)
print("Columns:", df_white_raw.columns.tolist())

print("\nAUDIT STRUKTUR")
print(
    "Nama dan urutan kolom sama:",
    df_red_raw.columns.equals(df_white_raw.columns)
)

print(
    "Kolom khusus red wine:",
    sorted(set(df_red_raw.columns) - set(df_white_raw.columns))
)

print(
    "Kolom khusus white wine:",
    sorted(set(df_white_raw.columns) - set(df_red_raw.columns))
)

print(
    "\nTotal baris sebelum penggabungan:",
    len(df_red_raw) + len(df_white_raw)
)

In [ ]:
# Audit struktur dataset red dan white wine

print("=" * 50)
print("STRUKTUR RED WINE")
print("=" * 50)
df_red_raw.info()

print("\n" + "=" * 50)
print("STRUKTUR WHITE WINE")
print("=" * 50)
df_white_raw.info()

In [ ]:
# Ringkasan tipe data kedua dataset

audit_dtypes = pd.DataFrame({
    "red_dtype": df_red_raw.dtypes.astype(str),
    "white_dtype": df_white_raw.dtypes.astype(str),
})

audit_dtypes["same_dtype"] = (
    audit_dtypes["red_dtype"] == audit_dtypes["white_dtype"]
)

display(audit_dtypes)

In [ ]:
# Audit missing values

missing_audit = pd.DataFrame({
    "red_missing": df_red_raw.isna().sum(),
    "white_missing": df_white_raw.isna().sum(),
})

missing_audit["red_missing_pct"] = (
    missing_audit["red_missing"]
    / len(df_red_raw) * 100
)

missing_audit["white_missing_pct"] = (
    missing_audit["white_missing"]
    / len(df_white_raw) * 100
)

display(missing_audit)

print(
    "\nTotal missing values red wine:",
    int(df_red_raw.isna().sum().sum())
)

print(
    "Total missing values white wine:",
    int(df_white_raw.isna().sum().sum())
)

In [ ]:
# Menghitung duplikat pada setiap dataset sumber

duplicate_red = df_red_raw.duplicated().sum()
duplicate_white = df_white_raw.duplicated().sum()

duplicate_audit_raw = pd.DataFrame({
    "dataset": ["Red wine", "White wine"],
    "total_rows": [len(df_red_raw), len(df_white_raw)],
    "duplicate_rows": [duplicate_red, duplicate_white],
})

duplicate_audit_raw["duplicate_pct"] = (
    duplicate_audit_raw["duplicate_rows"]
    / duplicate_audit_raw["total_rows"]
    * 100
)

display(duplicate_audit_raw)

In [ ]:
# Melihat semua baris yang termasuk dalam kelompok duplikat

red_duplicates = df_red_raw[
    df_red_raw.duplicated(keep=False)
].sort_values(by=df_red_raw.columns.tolist())

white_duplicates = df_white_raw[
    df_white_raw.duplicated(keep=False)
].sort_values(by=df_white_raw.columns.tolist())

print("Jumlah baris dalam kelompok duplikat red wine:",
      len(red_duplicates))

print("Jumlah baris dalam kelompok duplikat white wine:",
      len(white_duplicates))

In [ ]:
# Membuat salinan agar data sumber tetap tidak berubah

df_red_audit = df_red_raw.copy()
df_white_audit = df_white_raw.copy()

# Menambahkan label jenis wine
df_red_audit["wine_type"] = 0
df_white_audit["wine_type"] = 1

# Menggabungkan kedua dataset
df_combined = pd.concat(
    [df_red_audit, df_white_audit],
    ignore_index=True
)

print("Ukuran red wine   :", df_red_audit.shape)
print("Ukuran white wine :", df_white_audit.shape)
print("Ukuran gabungan   :", df_combined.shape)

display(df_combined.head())

In [ ]:
# Memastikan distribusi label wine_type

wine_type_counts = (
    df_combined["wine_type"]
    .value_counts()
    .rename(index={
        0: "Red wine",
        1: "White wine"
    })
)

wine_type_summary = pd.DataFrame({
    "count": wine_type_counts,
    "percentage": wine_type_counts / len(df_combined) * 100
})

display(wine_type_summary)

In [ ]:
# Audit duplikat pada dataset gabungan

# Duplikat identik pada seluruh kolom, termasuk wine_type
duplicate_all_columns = int(
    df_combined.duplicated().sum()
)

# Duplikat berdasarkan fitur dan quality,
# tanpa mempertimbangkan wine_type
feature_cols = [
    col for col in df_combined.columns
    if col != "wine_type"
]

duplicate_without_wine_type = int(
    df_combined.duplicated(
        subset=feature_cols
    ).sum()
)

print("AUDIT DUPLIKAT DATASET GABUNGAN")
print("=" * 50)

print(
    "Duplikat seluruh kolom:",
    duplicate_all_columns
)

print(
    "Duplikat fitur dan quality tanpa wine_type:",
    duplicate_without_wine_type
)

In [ ]:
# Mengidentifikasi kelompok fitur dan quality
# yang muncul pada lebih dari satu jenis wine

cross_type_groups = (
    df_combined
    .groupby(feature_cols)["wine_type"]
    .nunique()
)

cross_type_groups = cross_type_groups[
    cross_type_groups > 1
]

print(
    "Jumlah kelompok dengan wine_type berbeda:",
    len(cross_type_groups)
)

display(cross_type_groups)

In [ ]:
# Statistik deskriptif dataset gabungan dan masing-masing jenis wine

print("=" * 60)
print("STATISTIK DESKRIPTIF DATASET GABUNGAN")
print("=" * 60)
display(df_combined.describe().T)

print("\n" + "=" * 60)
print("STATISTIK DESKRIPTIF RED WINE")
print("=" * 60)
display(df_red_raw.describe().T)

print("\n" + "=" * 60)
print("STATISTIK DESKRIPTIF WHITE WINE")
print("=" * 60)
display(df_white_raw.describe().T)

In [ ]:
# Memeriksa nilai non-finite pada kolom numerik

numeric_cols = df_combined.select_dtypes(
    include=np.number
).columns

non_finite_audit = pd.DataFrame({
    "positive_infinity": np.isposinf(
        df_combined[numeric_cols]
    ).sum(),
    "negative_infinity": np.isneginf(
        df_combined[numeric_cols]
    ).sum(),
})

display(non_finite_audit)

In [ ]:
# Analisis distribusi quality

# 1. Memeriksa nilai unik dan rentang quality
print("Nilai quality yang ditemukan:")
print(sorted(df_combined["quality"].dropna().unique()))

print("\nNilai minimum quality:",
      df_combined["quality"].min())

print("Nilai maksimum quality:",
      df_combined["quality"].max())


# 2. Menghitung frekuensi dan persentase quality
quality_counts = (
    df_combined["quality"]
    .value_counts()
    .sort_index()
)

quality_summary = pd.DataFrame({
    "count": quality_counts,
    "percentage": quality_counts / len(df_combined) * 100
})

print("\nDistribusi quality keseluruhan:")
display(quality_summary)


# 3. Membandingkan quality berdasarkan jenis wine
quality_by_type = pd.crosstab(
    df_combined["quality"],
    df_combined["wine_type"],
    margins=True
)

quality_by_type = quality_by_type.rename(
    columns={
        0: "Red wine",
        1: "White wine"
    }
)

print("\nDistribusi quality berdasarkan jenis wine:")
display(quality_by_type)

In [ ]:

# Ringkasan kondisi dataset sebelum cleaning

audit_summary = pd.DataFrame({
    "metric": [
        "Jumlah baris red wine",
        "Jumlah baris white wine",
        "Total baris gabungan",
        "Jumlah kolom gabungan",
        "Missing values gabungan",
        "Duplikat semua kolom",
        "Duplikat tanpa wine_type",
        "Kelompok lintas jenis wine",
    ],
    "value": [
        len(df_red_raw),
        len(df_white_raw),
        len(df_combined),
        len(df_combined.columns),
        int(df_combined.isna().sum().sum()),
        int(df_combined.duplicated().sum()),
        int(df_combined.duplicated(subset=feature_cols).sum()),
        len(cross_type_groups),
    ]
})

display(audit_summary)


In [ ]:

# Evaluasi duplikat sebelum cleaning

total_rows = len(df_combined)
duplicate_rows = int(df_combined.duplicated().sum())
duplicate_percentage = (
    duplicate_rows / total_rows * 100
    if total_rows > 0 else 0
)

print("Total baris sebelum cleaning:", total_rows)
print("Jumlah baris duplikat:", duplicate_rows)
print(f"Persentase duplikat: {duplicate_percentage:.2f}%")

# Membandingkan jumlah baris setelah duplikat persis dihapus
df_no_exact_duplicates = df_combined.drop_duplicates()

print(
    "Jumlah baris setelah simulasi penghapusan duplikat:",
    len(df_no_exact_duplicates)
)
print(
    "Jumlah baris yang berkurang:",
    total_rows - len(df_no_exact_duplicates)
)


In [ ]:

# Evaluasi duplikat lintas jenis wine

# 1. Menghitung kelompok karakteristik yang muncul
#    pada kedua jenis wine
print(
    "Jumlah kelompok lintas jenis wine:",
    len(cross_type_groups)
)

# 2. Menghitung jumlah kelompok berdasarkan kombinasi
#    seluruh fitur selain wine_type
cross_type_counts = (
    df_combined
    .groupby(feature_cols, dropna=False)["wine_type"]
    .agg(["count", "nunique"])
)

# Mengambil kelompok yang memiliki lebih dari satu label wine_type
cross_type_conflicts = cross_type_counts[
    cross_type_counts["nunique"] > 1
]

print(
    "Jumlah kelompok dengan label wine_type berbeda:",
    len(cross_type_conflicts)
)

# 3. Menampilkan contoh baris yang termasuk kelompok tersebut
if not cross_type_conflicts.empty:
    cross_type_examples = (
        df_combined
        .set_index(feature_cols)
        .loc[cross_type_conflicts.index]
        .reset_index()
    )

    display(cross_type_examples.head(10))
else:
    print("Tidak ditemukan kelompok lintas jenis wine.")


In [ ]:

# Evaluasi dampak penghapusan duplikat persis

df_no_exact_duplicates = df_combined.drop_duplicates().copy()

print("Jumlah data sebelum cleaning:", len(df_combined))
print("Jumlah data setelah simulasi:", len(df_no_exact_duplicates))
print("Jumlah baris yang dihapus:",
      len(df_combined) - len(df_no_exact_duplicates))

# Membandingkan distribusi wine_type
print("\nDistribusi wine_type sebelum cleaning:")
display(df_combined["wine_type"].value_counts().sort_index())

print("\nDistribusi wine_type setelah simulasi:")
display(
    df_no_exact_duplicates["wine_type"]
    .value_counts()
    .sort_index()
)

# Membandingkan distribusi quality
quality_comparison = pd.concat(
    [
        df_combined["quality"].value_counts().sort_index().rename("before"),
        df_no_exact_duplicates["quality"].value_counts().sort_index().rename("after"),
    ],
    axis=1
).fillna(0).astype(int)

print("\nDistribusi quality sebelum dan setelah simulasi:")
display(quality_comparison)


In [ ]:

# Cleaning: menghapus duplikat persis

df_clean = df_combined.drop_duplicates().copy()

# Memastikan hasil cleaning sesuai dengan simulasi
assert len(df_clean) == 5320
assert df_clean.duplicated().sum() == 0

# Memeriksa missing values
print("Missing values setelah cleaning:")
print(df_clean.isna().sum().sum())

# Memeriksa duplikat tersisa
print("\nDuplikat persis setelah cleaning:")
print(df_clean.duplicated().sum())

# Memeriksa distribusi jenis wine
print("\nDistribusi wine_type setelah cleaning:")
display(df_clean["wine_type"].value_counts().sort_index())

print("\nUkuran dataset hasil cleaning:", df_clean.shape)


In [ ]:

# Menyimpan dataset hasil cleaning

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH = PROCESSED_DIR / "winequality_cleaned.csv"

df_clean.to_csv(CLEAN_DATA_PATH, index=False)

print("Dataset berhasil disimpan.")
print("Lokasi:", CLEAN_DATA_PATH)
print("Ukuran dataset:", df_clean.shape)


## Kesimpulan Data Preparation

Pada tahap data preparation, dataset red wine dan white wine digabungkan menjadi satu dataset dengan penambahan variabel `wine_type` sebagai label jenis wine. Red wine diberi label `0`, sedangkan white wine diberi label `1`.

Dataset awal terdiri dari 6.497 baris dan 13 kolom. Berdasarkan hasil audit, ditemukan 1.177 baris duplikat persis atau sekitar 18,12% dari total data. Setelah dilakukan penghapusan duplikat persis, jumlah data menjadi 5.320 baris.

Hasil validasi menunjukkan bahwa dataset hasil cleaning tidak memiliki missing values maupun duplikat persis. Dataset kemudian disimpan dalam format CSV dengan nama `winequality_cleaned.csv` pada folder `data/processed/`.

Dataset ini selanjutnya akan digunakan untuk analisis eksplorasi data (*Exploratory Data Analysis* atau EDA) dan eksperimen Machine Learning. Penghapusan duplikat merupakan keputusan preprocessing yang perlu diperhatikan dalam interpretasi hasil model karena dapat mengubah frekuensi pengamatan pada setiap kelas.